# Pilotage patrimonial global

Ce notebook fonctionne uniquement avec les valeurs saisies localement. Il ne passe aucun ordre.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.patrimoine import (
    compute_allocation,
    compute_projection,
    generate_patrimoine_report,
    load_patrimoine_config,
)

config = load_patrimoine_config(PROJECT_ROOT / 'config' / 'patrimoine.yaml')

In [2]:
# Remplis ces valeurs avant d'exécuter le notebook
valeurs_enveloppes = {
    'pea': 0,        # valeur actuelle du PEA en euros
    'livret_a': 0,   # valeur actuelle du Livret A en euros
    'ldds': 0,       # valeur actuelle du LDDS en euros
    'cto': 0,        # laisser à 0 si non ouvert
    'assurance_vie': 0,  # laisser à 0 si non ouvert
}

In [3]:
allocation = compute_allocation(valeurs_enveloppes, config)
resultats = generate_patrimoine_report(
    config, valeurs_enveloppes, PROJECT_ROOT / 'reports' / 'patrimoine_report.md'
)

enveloppes_df = pd.DataFrame([
    {
        'label': envelope['label'],
        'type': envelope['type'],
        'valeur_eur': valeurs_enveloppes.get(envelope['id'], 0),
        'actif': envelope['actif'],
    }
    for envelope in config['enveloppes']
])
allocation_df = pd.DataFrame({
    'actuelle_pct': allocation['current_pct'],
    'cible_pct': allocation['target_pct'],
    'derive_pct': allocation['drift_pct'],
})
projection_df = pd.DataFrame(resultats['projection']['scenarios']).T

display(enveloppes_df)
display(allocation_df)
display(projection_df)
print(resultats['recommendation'])
print(f"Rapport généré : {resultats['report_path']}")

,label,type,valeur_eur,actif
0,PEA,actions,0,True
1,Livret A,liquidites,0,True
2,LDDS,liquidites,0,True
3,CTO,actions,0,False
4,Assurance vie,mixte,0,False


,actuelle_pct,cible_pct,derive_pct
actions,0.0,100.0,-100.0
liquidites,0.0,0.0,0.0
mixte,0.0,0.0,0.0


,rendement_annuel_pct,valeur_finale_eur,total_verse_eur,plus_value_eur,rendement_total_pct
pessimiste,4.0,277619.761753,144000.0,133619.761753,92.791501
central,7.0,487988.398310,144000.0,343988.398310,238.880832
optimiste,10.0,904195.169918,144000.0,760195.169918,527.913312


Dérive détectée sur actions. Diriger le versement vers PEA.
Rapport généré : C:\Users\degar\OneDrive - De Vinci Higher Education (DVHE)\Perso\Projet Investissement\personal-quant-lab\reports\patrimoine_report.md
